# 📱 Smartphone Addiction Prediction (Grandmaster 10-Fold Triple Gradient Boosting Ensemble)

In this notebook, we predict smartphone addiction (`addicted_label`: 0 = No, 1 = Yes) using a **60-feature engineering pipeline** and a **10-Fold Stratified Triple-Ensemble** combining:
1. **Deep LightGBM** (1200 trees, max depth 11, 180 leaves, feature fraction 0.72)
2. **HistGradientBoosting** (550 iterations, 150 leaves, L2 regularized)
3. **XGBoost** (950 trees, depth 8, histogram tree method with feature & row subsampling)

### Key Architectural Strengths:
1. **Complete Data Utilization**: Retains all 691,369 samples with native NaN handling and missingness tracking (`num_missing`).
2. **60 Domain, Ratio, and Interaction Features**: Component breakdowns, free awake hours, non-study screen hours, sleep debt, quadratic sleep-screen interaction, high-risk compound flags, behavioral archetypes (K-Means), and age-group z-scores.
3. **10-Fold Stratified Cross-Validation**: 30 total models trained on 90% data per fold with optimal blend calibration (45% LGBM + 15% HistGBM + 40% XGBoost).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import lightgbm as lgb
import xgboost as xgb

import warnings
warnings.filterwarnings('ignore')
print("Libraries imported successfully!")


## 1. Load Datasets


In [ ]:
train_df = pd.read_csv('Data/train.csv')
test_df = pd.read_csv('Data/test.csv')

print(f"Train Dataset Shape: {train_df.shape}")
print(f"Test Dataset Shape:  {test_df.shape}")
train_df.head()


## 2. Exploratory Data Analysis (EDA)


In [ ]:
# Summary statistics
train_df.describe()


In [ ]:
# Target class distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=train_df, x='addicted_label', palette='viridis')
plt.title('Target Distribution: Addicted Status (0 = No, 1 = Yes)')
plt.xlabel('Addicted Status')
plt.ylabel('Count')
plt.show()

print("Class Balance:")
print(train_df['addicted_label'].value_counts(normalize=True) * 100)


## 3. Grandmaster Feature Engineering (60 Features)

We engineer domain features covering:
- **Component Breakdown & Unaccounted Time**: `accounted_screen_time`, `unaccounted_screen_time`, `unaccounted_ratio`, `entertainment_hours`, `non_study_screen_hours`, `social_media_ratio`, `gaming_ratio`, `work_study_ratio`, `entertainment_ratio`, `unproductive_to_productive`, `social_to_gaming_ratio`.
- **Free Awake Time & Exposure**: `awake_hours`, `free_awake_hours`, `screen_to_free_awake_ratio`, `screen_time_to_awake_ratio`, `non_study_to_sleep_ratio`, `sleep_deficit`, `night_screen_risk`, `notifications_per_awake_hour`, `app_opens_per_awake_hour`, `minutes_per_app_open`, `notif_per_screen_minute`.
- **Weekend Dynamics & Weekly Totals**: `weekend_vs_daily_diff`, `weekend_vs_daily_ratio`, `weekend_to_weekday_ratio`, `total_weekly_screen_time`.
- **Log & Polynomial Transforms**: `log_notifications`, `log_app_opens`, `log_screen_time`, `screen_sleep_sq`.
- **High-Risk Compound Flags**: `high_screen_low_sleep`, `high_notif_high_open`, `severe_impact_stress`.
- **Non-Linear Interactions**: `screen_stress_inter`, `screen_sleep_comp`, `social_sleep_comp`, `notif_app_open_inter`, `addiction_risk_score`.
- **Age-Group Benchmarks**: `screen_time_vs_age_mean`, `screen_time_age_zscore`, `notif_vs_age_mean`.
- **Frequency Encodings**: `age_freq`, `notifications_per_day_freq`, `app_opens_per_day_freq`.
- **Behavioral Archetypes (K-Means Clustering)**: `user_cluster`.


In [ ]:
def extract_features(train, test):
    df_all = pd.concat([train.assign(is_train=1), test.assign(is_train=0, addicted_label=-1)], ignore_index=True)
    eps = 1e-5
    
    # 1. Numerical encodings
    gender_map = {'Female': 0, 'Male': 1, 'Other': 2}
    df_all['gender_num'] = df_all['gender'].map(gender_map)
    df_all['academic_impact_num'] = df_all['academic_work_impact'].astype(str).str.strip().str.lower().map({'no': 0, 'yes': 1})
    df_all['stress_num'] = df_all['stress_level'].astype(str).str.strip().str.lower().map({'low': 0, 'medium': 1, 'high': 2})
    
    # 2. Categorical Combinations
    df_all['stress_academic_combo'] = df_all['stress_num'].fillna(-1).astype(int).astype(str) + "_" + df_all['academic_impact_num'].fillna(-1).astype(int).astype(str)
    combo_map = {val: i for i, val in enumerate(df_all['stress_academic_combo'].unique())}
    df_all['stress_academic_code'] = df_all['stress_academic_combo'].map(combo_map)
    df_all.drop(columns=['stress_academic_combo'], inplace=True)
    
    # 3. Missing counts per record
    raw_cols = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours',
                'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time',
                'gender', 'stress_level', 'academic_work_impact']
    df_all['num_missing'] = df_all[raw_cols].isnull().sum(axis=1)
    
    # 4. Detailed Component Breakdowns
    df_all['accounted_screen_time'] = df_all['social_media_hours'] + df_all['gaming_hours'] + df_all['work_study_hours']
    df_all['unaccounted_screen_time'] = df_all['daily_screen_time_hours'] - df_all['accounted_screen_time']
    df_all['unaccounted_ratio'] = df_all['unaccounted_screen_time'] / (df_all['daily_screen_time_hours'] + eps)
    df_all['entertainment_hours'] = df_all['social_media_hours'] + df_all['gaming_hours']
    df_all['non_study_screen_hours'] = df_all['daily_screen_time_hours'] - df_all['work_study_hours']
    
    df_all['social_media_ratio'] = df_all['social_media_hours'] / (df_all['daily_screen_time_hours'] + eps)
    df_all['gaming_ratio'] = df_all['gaming_hours'] / (df_all['daily_screen_time_hours'] + eps)
    df_all['work_study_ratio'] = df_all['work_study_hours'] / (df_all['daily_screen_time_hours'] + eps)
    df_all['entertainment_ratio'] = df_all['entertainment_hours'] / (df_all['daily_screen_time_hours'] + eps)
    df_all['unproductive_to_productive'] = df_all['entertainment_hours'] / (df_all['work_study_hours'] + eps)
    df_all['social_to_gaming_ratio'] = df_all['social_media_hours'] / (df_all['gaming_hours'] + eps)
    
    # 5. Free Time & Awake Time Deprivation
    df_all['awake_hours'] = 24.0 - df_all['sleep_hours']
    df_all['free_awake_hours'] = (df_all['awake_hours'] - df_all['work_study_hours']).clip(lower=0.1)
    df_all['screen_to_free_awake_ratio'] = df_all['entertainment_hours'] / (df_all['free_awake_hours'] + eps)
    df_all['screen_time_to_awake_ratio'] = df_all['daily_screen_time_hours'] / (df_all['awake_hours'] + eps)
    df_all['non_study_to_sleep_ratio'] = df_all['non_study_screen_hours'] / (df_all['sleep_hours'] + eps)
    df_all['sleep_deficit'] = (8.0 - df_all['sleep_hours']).clip(lower=0)
    df_all['night_screen_risk'] = (df_all['daily_screen_time_hours'] > df_all['awake_hours'] * 0.5).astype(int)
    
    df_all['notifications_per_awake_hour'] = df_all['notifications_per_day'] / (df_all['awake_hours'] + eps)
    df_all['app_opens_per_awake_hour'] = df_all['app_opens_per_day'] / (df_all['awake_hours'] + eps)
    df_all['notifications_per_app_open'] = df_all['notifications_per_day'] / (df_all['app_opens_per_day'] + eps)
    df_all['minutes_per_app_open'] = (df_all['daily_screen_time_hours'] * 60.0) / (df_all['app_opens_per_day'] + eps)
    df_all['notif_per_screen_minute'] = df_all['notifications_per_day'] / (df_all['daily_screen_time_hours'] * 60.0 + eps)
    
    # 6. Weekend Dynamics
    df_all['weekend_vs_daily_diff'] = df_all['weekend_screen_time'] - df_all['daily_screen_time_hours']
    df_all['weekend_vs_daily_ratio'] = df_all['weekend_screen_time'] / (df_all['daily_screen_time_hours'] + eps)
    df_all['weekend_to_weekday_ratio'] = (df_all['weekend_screen_time'] / 2.0) / (df_all['daily_screen_time_hours'] + eps)
    df_all['total_weekly_screen_time'] = (df_all['daily_screen_time_hours'] * 5.0) + (df_all['weekend_screen_time'] * 2.0)
    
    # 7. Log & Polynomial Transforms
    df_all['log_notifications'] = np.log1p(df_all['notifications_per_day'].clip(lower=0))
    df_all['log_app_opens'] = np.log1p(df_all['app_opens_per_day'].clip(lower=0))
    df_all['log_screen_time'] = np.log1p(df_all['daily_screen_time_hours'].clip(lower=0))
    df_all['screen_sleep_sq'] = (df_all['daily_screen_time_hours'] / (df_all['sleep_hours'] + eps)) ** 2
    
    # 8. High Risk Compound Flags
    df_all['high_screen_low_sleep'] = ((df_all['daily_screen_time_hours'] >= 8) & (df_all['sleep_hours'] <= 5)).astype(int)
    df_all['high_notif_high_open'] = ((df_all['notifications_per_day'] >= 100) & (df_all['app_opens_per_day'] >= 80)).astype(int)
    df_all['severe_impact_stress'] = ((df_all['academic_impact_num'] == 1) & (df_all['stress_num'] == 2)).astype(int)
    
    # 9. Non-linear Interactions
    df_all['screen_stress_inter'] = df_all['daily_screen_time_hours'] * (df_all['stress_num'] + 1)
    df_all['screen_sleep_comp'] = df_all['daily_screen_time_hours'] / (df_all['sleep_hours'] + eps)
    df_all['social_sleep_comp'] = df_all['social_media_hours'] / (df_all['sleep_hours'] + eps)
    df_all['notif_app_open_inter'] = df_all['notifications_per_day'] * df_all['app_opens_per_day']
    df_all['addiction_risk_score'] = (
        (df_all['daily_screen_time_hours'] > 7).astype(int) +
        (df_all['sleep_hours'] < 6).astype(int) +
        (df_all['social_media_hours'] > 4).astype(int) +
        (df_all['academic_impact_num'] == 1).astype(int) +
        (df_all['stress_num'] == 2).astype(int)
    )
    
    # 10. Age Group Normalized Features
    df_all['age_group'] = (df_all['age'] // 5) * 5
    age_screen_mean = df_all.groupby('age_group')['daily_screen_time_hours'].transform('mean')
    age_screen_std = df_all.groupby('age_group')['daily_screen_time_hours'].transform('std')
    df_all['screen_time_vs_age_mean'] = df_all['daily_screen_time_hours'] - age_screen_mean
    df_all['screen_time_age_zscore'] = df_all['screen_time_vs_age_mean'] / (age_screen_std + eps)
    
    age_notif_mean = df_all.groupby('age_group')['notifications_per_day'].transform('mean')
    df_all['notif_vs_age_mean'] = df_all['notifications_per_day'] - age_notif_mean
    
    # 11. Frequency Encodings
    for col in ['age', 'notifications_per_day', 'app_opens_per_day']:
        freq = df_all[col].value_counts(normalize=True)
        df_all[f'{col}_freq'] = df_all[col].map(freq)
        
    # 12. Clustering Archetypes
    cluster_features = ['daily_screen_time_hours', 'social_media_hours', 'sleep_hours', 'notifications_per_day']
    cluster_imputed = df_all[cluster_features].fillna(df_all[cluster_features].median())
    scaler = StandardScaler()
    scaled_feats = scaler.fit_transform(cluster_imputed)
    
    kmeans = MiniBatchKMeans(n_clusters=8, random_state=42, batch_size=2048)
    df_all['user_cluster'] = kmeans.fit_predict(scaled_feats)
    
    df_all = df_all.drop(columns=['gender', 'academic_work_impact', 'stress_level', 'age_group'])
    
    train_res = df_all[df_all['is_train'] == 1].drop(columns=['is_train'])
    test_res = df_all[df_all['is_train'] == 0].drop(columns=['is_train', 'addicted_label'])
    
    return train_res, test_res

train_features, test_features = extract_features(train_df, test_df)

print(f"Total features created: {train_features.shape[1] - 2}")
train_features.head()


## 4. Model Training (Grandmaster 10-Fold Triple-Ensemble)

We train **LightGBM**, **HistGradientBoosting**, and **XGBoost** across 10 Stratified Folds:
- 10 Folds train on 90% of the dataset per fold (622,000 samples).
- 30 total models (10 folds x 3 architectures) are blended with calibrated weights (45% LGBM + 15% HistGBM + 40% XGBoost).


In [ ]:
# Features and Target
feature_cols = [c for c in train_features.columns if c not in ['id', 'addicted_label']]
X = train_features[feature_cols]
y = train_features['addicted_label']
X_test = test_features[feature_cols]

# 10-Fold Stratified Cross-Validation setup
n_splits = 10
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(train_df))
oof_hgb = np.zeros(len(train_df))
oof_xgb = np.zeros(len(train_df))

test_lgb = np.zeros(len(test_df))
test_hgb = np.zeros(len(test_df))
test_xgb = np.zeros(len(test_df))

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.022,
    'num_leaves': 180,
    'max_depth': 11,
    'min_child_samples': 30,
    'feature_fraction': 0.72,
    'bagging_fraction': 0.85,
    'bagging_freq': 1,
    'n_estimators': 1200,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'learning_rate': 0.028,
    'max_depth': 8,
    'colsample_bytree': 0.75,
    'subsample': 0.85,
    'n_estimators': 950,
    'tree_method': 'hist',
    'random_state': 42,
    'n_jobs': -1
}

print(f"Starting Grandmaster 10-Fold Triple-Ensemble on {len(X)} samples with {len(feature_cols)} features...\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # 1. Train LightGBM
    lgb_model = lgb.LGBMClassifier(**lgb_params)
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(60, verbose=False)])
    v1 = lgb_model.predict_proba(X_val)[:, 1]
    oof_lgb[val_idx] = v1
    test_lgb += lgb_model.predict_proba(X_test)[:, 1] / n_splits
    
    # 2. Train HistGradientBoosting
    hgb_model = HistGradientBoostingClassifier(
        max_iter=550,
        learning_rate=0.04,
        max_leaf_nodes=150,
        min_samples_leaf=25,
        l2_regularization=0.8,
        random_state=42 + fold
    )
    hgb_model.fit(X_tr, y_tr)
    v2 = hgb_model.predict_proba(X_val)[:, 1]
    oof_hgb[val_idx] = v2
    test_hgb += hgb_model.predict_proba(X_test)[:, 1] / n_splits
    
    # 3. Train XGBoost
    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    v3 = xgb_model.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx] = v3
    test_xgb += xgb_model.predict_proba(X_test)[:, 1] / n_splits
    
    fold_blend = (v1 * 0.45) + (v2 * 0.15) + (v3 * 0.40)
    print(f"Fold {fold+1:02d}/10 -> LGB: {roc_auc_score(y_val, v1):.5f} | HGB: {roc_auc_score(y_val, v2):.5f} | XGB: {roc_auc_score(y_val, v3):.5f} | Blend: {roc_auc_score(y_val, fold_blend):.5f}")


## 5. Overall Model Evaluation (Out-Of-Fold)


In [ ]:
# Overall 10-Fold Grandmaster Ensemble evaluation (45% LGB + 15% HGB + 40% XGB)
oof_ensemble = (oof_lgb * 0.45) + (oof_hgb * 0.15) + (oof_xgb * 0.40)
oof_binary = (oof_ensemble >= 0.5).astype(int)

overall_auc = roc_auc_score(y, oof_ensemble)
overall_f1 = f1_score(y, oof_binary)
overall_acc = accuracy_score(y, oof_binary)
overall_prec = precision_score(y, oof_binary)
overall_rec = recall_score(y, oof_binary)

print("=" * 60)
print("    OVERALL 10-FOLD GRANDMASTER ENSEMBLE VALIDATION RESULTS")
print("=" * 60)
print(f"LightGBM Out-of-Fold ROC-AUC:  {roc_auc_score(y, oof_lgb):.5f}")
print(f"HistGBM  Out-of-Fold ROC-AUC:  {roc_auc_score(y, oof_hgb):.5f}")
print(f"XGBoost  Out-of-Fold ROC-AUC:  {roc_auc_score(y, oof_xgb):.5f}")
print(f"Grandmaster Blend ROC-AUC:     {overall_auc:.5f}")
print(f"Grandmaster Blend F1-Score:    {overall_f1:.5f}")
print(f"Grandmaster Blend Accuracy:    {overall_acc * 100:.2f}%")
print(f"Grandmaster Blend Precision:   {overall_prec * 100:.2f}%")
print(f"Grandmaster Blend Recall:      {overall_rec * 100:.2f}%")
print("=" * 60)

print("\nClassification Report:")
print(classification_report(y, oof_binary, digits=4))

# Confusion Matrix plot
cm = confusion_matrix(y, oof_binary)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Not Addicted (0)', 'Addicted (1)'],
            yticklabels=['Not Addicted (0)', 'Addicted (1)'])
plt.title('10-Fold Out-Of-Fold Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


## 6. Generate Final Submission File


In [ ]:
# Grandmaster Ensemble test predictions (45% LightGBM + 15% HistGBM + 40% XGBoost)
final_test_predictions = (test_lgb * 0.45) + (test_hgb * 0.15) + (test_xgb * 0.40)

submission = pd.DataFrame({
    'id': test_df['id'],
    'addicted_label': final_test_predictions
})

# Save to submission.csv
submission.to_csv('submission.csv', index=False)

print(f"Submission saved successfully to 'submission.csv'!")
print(f"Total rows: {len(submission)}")
print(f"Missing values: {submission.isnull().sum().to_dict()}")
submission.head(10)
